In [1]:
# autoreload
%load_ext autoreload
%autoreload 2

In [2]:
# Cell 1.5: FORCE LOAD environment variables from .envrc file - RUN THIS FIRST!
import os

print("🔧 FORCE Loading environment variables from .envrc...")
try:
    with open('../.envrc', 'r') as f:
        for line in f:
            line = line.strip()
            if line.startswith('export ') and '=' in line:
                # Parse export VAR='value' or export VAR=value
                var_assignment = line[7:]  # Remove 'export '
                key, value = var_assignment.split('=', 1)
                # Remove quotes if present
                value = value.strip('"').strip("'")
                os.environ[key] = value
                print(f"✅ {key} loaded")

    print("\n🔑 Environment status:")
    print(f"✓ OPENAI_API_KEY: {'SET' if os.getenv('OPENAI_API_KEY') else 'NOT SET'}")
    print(f"✓ XAI_API_KEY: {'SET' if os.getenv('XAI_API_KEY') else 'NOT SET'}")
    print(f"✓ POLYGON_API_KEY: {'SET' if os.getenv('POLYGON_API_KEY') else 'NOT SET'}")
    print(f"✓ SEC_IDENTITY_EMAIL: {'SET' if os.getenv('SEC_IDENTITY_EMAIL') else 'NOT SET'}")

    # Check critical keys
    missing_keys = []
    if not os.getenv('OPENAI_API_KEY'):
        missing_keys.append('OPENAI_API_KEY')
    if not os.getenv('SEC_IDENTITY_EMAIL'):
        missing_keys.append('SEC_IDENTITY_EMAIL')
    
    if missing_keys:
        print(f"\n💥 CRITICAL: {', '.join(missing_keys)} NOT LOADED!")
        print("🚨 Some features will not work:")
        if 'OPENAI_API_KEY' in missing_keys:
            print("   - Agents will fail with 'OPENAI_API_KEY is not set' errors")
        if 'SEC_IDENTITY_EMAIL' in missing_keys:
            print("   - SEC filing analysis (get_sec_filing) will fail")
        print("\n✅ SOLUTION: Close VS Code, run 'direnv allow' in terminal, then 'code .'")
    else:
        print("\n✅ SUCCESS: All critical API keys loaded! All features will work.")

except Exception as e:
    print(f"❌ Error: {e}")

🔧 FORCE Loading environment variables from .envrc...
✅ OPENAI_API_KEY loaded
✅ POLYGON_API_KEY loaded
✅ XAI_API_KEY loaded
✅ SEC_IDENTITY_EMAIL loaded

🔑 Environment status:
✓ OPENAI_API_KEY: SET
✓ XAI_API_KEY: SET
✓ POLYGON_API_KEY: SET
✓ SEC_IDENTITY_EMAIL: SET

✅ SUCCESS: All critical API keys loaded! All features will work.


In [3]:
import sys                                                                                                                                                          
sys.path.append('..')   
# SimpleAgent: No memory, treats each question independently
from stocks_agent.simple_agent import SimpleAgent

simple = SimpleAgent(model="gpt-5.4-mini")
print("✓ SimpleAgent initialized")

✓ SimpleAgent initialized


In [4]:
# Test basic query
response = await simple.ask("Compare Netflix and Disney")
print(response)

🤖 Model: gpt-5.4-mini

Here’s a concise side-by-side of **Netflix (NFLX)** vs **Disney (DIS)** based on the latest data I pulled.

## Quick take
- **Netflix** looks like the stronger **pure streaming / cash-flow compounder**: higher margins, higher ROE, stronger free cash flow, and better revenue/EPS growth.
- **Disney** looks like the more **diversified media + parks + IP** play: cheaper valuation, stronger analyst rating, but lower margins and weaker recent EPS growth.

## Snapshot

| Metric | Netflix (NFLX) | Disney (DIS) |
|---|---:|---:|
| Price | $81.49 | $101.86 |
| Market cap | $343.1B | $176.9B |
| Trailing P/E | 26.3x | 16.3x |
| Forward P/E | 21.2x | 13.6x |
| Price/Sales | 7.32x | 1.82x |
| Operating margin | 32.3% | 15.5% |
| Profit margin | 28.5% | 11.5% |
| ROE | 48.5% | 11.0% |
| Revenue growth | 16.2% | 6.5% |
| Earnings growth | 86.4% | -29.8% |
| Free cash flow | $26.0B | $3.75B |
| Debt / equity | 53.8 | 41.1 |
| Analyst rating | Buy | Strong Buy |

## What stands o

In [5]:
# ConversationAgent: Remembers context, auto-tracks tickers, handles follow-ups
  # Cell 1: Import and initialize ConversationAgent
from stocks_agent.conversation_agent import ConversationAgent

conv = ConversationAgent(track_tickers=True, model="gpt-5.4-mini")
print("✓ ConversationAgent initialized")


✓ ConversationAgent initialized


In [6]:
# Conv agent: Ask the same question about AAPL

response = await conv.ask("Tell me about the current opinion on the AI stocks?")
print(response)

Fetching 5000 news articles...


API calls: 100%|██████████| 5/5 [00:09<00:00,  1.99s/it]


Preprocessing documents...


Converting fields: 100%|██████████| 5000/5000 [00:00<00:00, 1277427.06it/s]


Building search index...
{'articles_indexed': 5000,
 'message': 'Index built with 5000 articles',
 'status': 'success'}
✓ Loaded 10879 companies from marketcap
✓ Loaded 10879 companies from pe_ratio
✓ Loaded 10879 companies from dividend
✓ Loaded 10879 companies from margin

Merging databases...
✓ Added P/E ratio column
✓ Added Dividend yield column
✓ Added Operating margin column

Final columns: ['Rank', 'Name', 'Symbol', 'marketcap', 'price (USD)', 'country', 'pe_ratio_ttm', 'dividend_yield_ttm', 'operating_margin_ttm']
{'available_columns': ['Rank',
                       'Name',
                       'Symbol',
                       'marketcap',
                       'price (USD)',
                       'country',
                       'pe_ratio_ttm',
                       'dividend_yield_ttm',
                       'operating_margin_ttm'],
 'databases_loaded': {'dividend': 10879,
                      'margin': 10879,
                      'marketcap': 10879,
               

/Users/realmistic/Documents/stocks-scoring-agent/notebooks/../stocks_agent/tools.py:721: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  div_df['dividend_yield_ttm'] = div_df['dividend_yield_ttm'] / 100.0
/Users/realmistic/Documents/stocks-scoring-agent/notebooks/../stocks_agent/tools.py:729: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  margin_df['operating_margin_ttm'] = margin_df['operating_margin_ttm']/100.0


The **current opinion on AI stocks is still positive overall, but much more selective than it was at the peak of the hype cycle**. The market is still rewarding companies tied to AI infrastructure, chips, servers, and data-center buildout, but investors are also rotating away from the most obvious mega-cap names and asking harder questions about valuation and durability. Recent coverage shows that **Nvidia remains central to the AI trade, but it’s no longer the only winner**; names like Dell, HPQ, QCOM, NTAP, and HPE have been getting attention as “second-derivative” AI beneficiaries, while some articles explicitly note a rotation out of flagship AI leaders into suppliers and infrastructure plays. citeturn0search0turn0search1turn0search5turn0search6

A few themes stand out:

- **AI is still a real earnings driver.** Recent market commentary says tech and AI earnings have been a major force behind market momentum, and several large-cap names tied to AI show strong profitability an

In [ ]:
response2 = await conv.ask("follow-up: what are the next earnings catalysts?")
print(response2)

In [ ]:
# StructuredAgent: Returns both text analysis + structured data dict for programmatic use

from stocks_agent.structured_agent import StructuredAgent
import json
structured = StructuredAgent(model="gpt-5.4-mini")
print("✓ StructuredAgent initialized")

In [ ]:
# Analyze AVGO
text, data = await structured.analyze('AVGO')

# show the resulting text
print("="*80)
print("TEXT RESPONSE:")
print(text)

In [ ]:
# show the structured data -- improved version with better formatting and handling of non-serializable data
print("\n" + "="*80)
print("STRUCTURED DATA:")
print(json.dumps(data, indent=2, default=str))

In [ ]:
# Free Agent: No running cost agent: use Ollama to run a local model (qwen3:32b) for free, no API key needed + call FREE tools (like get_company_info, get_eps_trend, etc.) to answer questions about stocks

from stocks_agent.free_agent import FreeAgent

# Setup: ollama pull qwen3:32b && ollama serve
agent = FreeAgent(model='qwen3:32b')
response = await agent.ask("What going on with Broadcom? Analyze vs. competitors")
print(agent.get_status())  # ✅ Connected to Ollama

In [ ]:
print(response)